In [ ]:
import pandas as pd
import json

    # Deduplicated and cleaned dataset
books_df = pd.read_csv(
    r"C:\Users\LOQ\project_books\.venv\datas\processed_books_dataset.csv",
    dtype={"google_id": "string"}
)

openlibrary_df = pd.read_csv(
    r"C:\Users\LOQ\project_books\.venv\datas\processing_data\Experiments\openlibrary_metadata.csv",
    dtype={"google_id": "string"}
)

In [6]:
openlibrary_df.head()

,google_id,title,openlibrary_work_key,openlibrary_description,openlibrary_subjects,match_method
0,_DYNAQAAMAAJ,The Face of Another,/works/OL8037368W,NaN,"[""Near and far eastern fiction (fictional work...",title_author
1,ggpgAAAAMAAJ,Generations of Winter,NaN,NaN,NaN,not_found
2,v7VPEQAAQBAJ,The Street of Crocodiles,/works/OL2510290W,This a magical book of short stories set in a ...,"[""Fiction"", ""Polish fiction"", ""Fantasy"", ""Poli...",title_author
3,FSTRAgAAQBAJ,Kokoro,/works/OL25366592W,NaN,[],isbn_13
4,qvk6DwAAQBAJ,Sense and Sensibility - Illustrated Edition,NaN,NaN,NaN,not_found


In [7]:
books_enriched = books_df.merge(
    openlibrary_df[
        [
            "google_id",
            "openlibrary_description",
            "openlibrary_subjects",
            "match_method"
        ]
    ],
    on="google_id",
    how="left"
)

In [11]:
import json

def parse_subjects(value):

    if pd.isna(value):
        return []

    try:
        subjects = json.loads(value)

        if isinstance(subjects, list):
            return subjects

    except (json.JSONDecodeError, TypeError):
        pass

    return []

books_enriched["openlibrary_categories"] = (
    books_enriched["openlibrary_subjects"]
    .apply(parse_subjects)
    .apply(lambda x: ", ".join(x))
)

In [12]:
def combine_categories(google_categories, openlibrary_categories):

    categories = []

    if pd.notna(google_categories):
        categories.extend(
            str(google_categories).split(",")
        )

    if pd.notna(openlibrary_categories):
        categories.extend(
            str(openlibrary_categories).split(",")
        )

    # Clean
    categories = [
        c.strip()
        for c in categories
        if c.strip()
    ]

    # Remove duplicates while preserving order
    categories = list(dict.fromkeys(categories))

    return ", ".join(categories)

In [13]:
generic_categories = {
    "fiction",
    "books",
    "literature",
    "general",
    "etc"
}

def prepare_categories_for_sbert(category_text, max_categories=10):

    if pd.isna(category_text):
        return ""

    categories = [
        c.strip()
        for c in str(category_text).split(",")
        if c.strip()
    ]

    # Remove duplicates
    categories = list(dict.fromkeys(categories))

    specific_categories = [
        c for c in categories
        if c.lower() not in generic_categories
    ]

    if specific_categories:
        categories = specific_categories
    else:
        categories = categories
    categories = categories[:max_categories]

    return ", ".join(categories)

In [14]:
books_enriched["categories_final"] = (
    books_enriched.apply(
        lambda row: combine_categories(
            row["categories"],
            row["openlibrary_categories"]
        ),
        axis=1
    )
)

In [15]:
books_enriched["categories_sbert"] = (
    books_enriched["categories_final"]
    .apply(prepare_categories_for_sbert)
)

In [16]:
category_counts = (
    books_enriched["categories"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print(category_counts.head(30))

categories
Fiction                          930
Juvenile Fiction                 292
History                          217
Biography & Autobiography        205
Drama                            155
Science                          148
Comics & Graphic Novels          141
Philosophy                       121
Psychology                       100
Poetry                            84
Literary Criticism                78
Juvenile Nonfiction               71
Adventure                         71
Education                         61
Art                               49
etc                               46
Business & Economics              45
Crime                             44
Family & Relationships            43
strips                            42
Detective and mystery stories     39
Children's stories                39
Social Science                    38
Children                          37
Comic books                       32
Self-Help                         32
Religion                   

In [17]:
for col in ["categories", "categories_final"]:
    values = books_enriched[col].astype("string").str.strip()
    print(
        col,
        "nulls:", books_enriched[col].isna().sum(),
        "blanks:", values.eq("").fillna(False).sum()
    )

categories nulls: 673 blanks: 0
categories_final nulls: 0 blanks: 576


In [18]:
categories = books_enriched["categories_sbert"].fillna("").astype(str)

category_counts = categories.map(
    lambda value: sum(bool(item.strip()) for item in value.split(","))
)

print(category_counts.value_counts().sort_index())
print("Books with zero categories:", (category_counts == 0).sum())

categories_sbert
0      576
1     3306
2      189
3       71
4       74
5       67
6       65
7       77
8       59
9       61
10     651
Name: count, dtype: int64
Books with zero categories: 576


In [19]:
top_category_books = (
    books_enriched
    .assign(category_count=category_counts)
    .sort_values("category_count", ascending=False)
)

print(
    top_category_books[
        ["title", "categories_sbert", "category_count"]
    ].head(10).to_string(index=False)
)

                    title                                                                                                                                                                            categories_sbert  category_count
              The Vampyre                                                   Poetry, Incest, Vampires, English Horror tales, Classic Literature, Horror, Short Stories, Fiction horror, Fiction ghost, English fiction              10
     The Three Musketeers                History, Swordsmen, France in fiction, Vietnamese language books, Swordplay, Adventure and adventurers, Spanish language materials, Romans, nouvelles, etc. pour la jeunesse              10
      The Pickwick Papers                          Men, Societies and clubs, Male friendship, Trials (Breach of promise), Social life and customs, Readers, Inglés, Travelers, Literatura inglesa, English literature              10
  How I Found Livingstone                                                   Desc

In [21]:
books_enriched["description_final"] = (
    books_enriched["description"]
    .fillna(
        books_enriched["openlibrary_description"]
    )
)

In [22]:
print(
    "Original descriptions missing:",
    books_enriched["description"].isna().sum()
)

print(
    "Final descriptions missing:",
    books_enriched["description_final"].isna().sum()
)

Original descriptions missing: 1079
Final descriptions missing: 1004


In [23]:
print(
    books_enriched[
        ["title", "categories_sbert"]
    ].head(30).to_string(index=False)
)

                                                     title                                                                                                                                                                            categories_sbert
                                       The Face of Another                                                                                                                                Near and far eastern fiction (fictional works by one author)
                                     Generations of Winter                                                                                                                                                                                     Fiction
                                  The Street of Crocodiles                                        Polish fiction, Fantasy, Polish Short stories, Crocodiles, Children's literature, Polish Science fiction, Families, Translations into English, Drama
            

In [24]:
books_enriched.to_csv(
    "books_enriched.csv",
    index=False
)

In [25]:
books_enriched.head(10)

,google_id,title,subtitle,authors,publisher,published_date,description,categories,page_count,language,...,clean_description,author_set,completeness,openlibrary_description,openlibrary_subjects,match_method,openlibrary_categories,categories_final,categories_sbert,description_final
0,_DYNAQAAMAAJ,The Face of Another,NaN,Kōbō Abe,Kodansha,1992,The Japanese novelist Kobo Abe has often been ...,Fiction,260.0,en,...,The Japanese novelist Kobo Abe has often been ...,{'kōbō abe'},5,NaN,"[""Near and far eastern fiction (fictional work...",title_author,Near and far eastern fiction (fictional works ...,"Fiction, Near and far eastern fiction (fiction...",Near and far eastern fiction (fictional works ...,The Japanese novelist Kobo Abe has often been ...
1,ggpgAAAAMAAJ,Generations of Winter,NaN,Василий Аксенов,Random House (NY),1994,"""Written in the great tradition of epic Russia...",Fiction,616.0,en,...,"""Written in the great tradition of epic Russia...",{'василий аксенов'},5,NaN,NaN,not_found,,Fiction,Fiction,"""Written in the great tradition of epic Russia..."
2,v7VPEQAAQBAJ,The Street of Crocodiles,NaN,Bruno Schulz,Rare Treasure Editions,2025-03-19T00:00:00Z,''The Street of Crocodiles'' by Bruno Schulz (...,Fiction,136.0,en,...,''The Street of Crocodiles'' by Bruno Schulz (...,{'bruno schulz'},7,This a magical book of short stories set in a ...,"[""Fiction"", ""Polish fiction"", ""Fantasy"", ""Poli...",title_author,"Fiction, Polish fiction, Fantasy, Polish Short...","Fiction, Polish fiction, Fantasy, Polish Short...","Polish fiction, Fantasy, Polish Short stories,...",''The Street of Crocodiles'' by Bruno Schulz (...
3,FSTRAgAAQBAJ,Kokoro,Hints and Echoes of Japanese Inner Life,Lafcadio Hearn,Tuttle Publishing,2012-12-20,The 15 classic essays collected in Kokoro exam...,Literary Collections,267.0,en,...,The 15 classic essays collected in Kokoro exam...,{'lafcadio hearn'},7,NaN,[],isbn_13,,Literary Collections,Literary Collections,The 15 classic essays collected in Kokoro exam...
4,qvk6DwAAQBAJ,Sense and Sensibility - Illustrated Edition,NaN,Jane Austen,Abela Publishing Ltd,2017-10-23,SENSE AND SENSIBILITY written by Jane Austen w...,Fiction,579.0,en,...,SENSE AND SENSIBILITY written by Jane Austen w...,{'jane austen'},7,NaN,NaN,not_found,,Fiction,Fiction,SENSE AND SENSIBILITY written by Jane Austen w...
5,XUtNEQAAQBAJ,The Final Problem,NaN,Arthur Conan Doyle,SAMPI Books,2025-03-08,"In ""The Final Problem"", Sherlock Holmes finds ...",Fiction,39.0,en,...,"In ""The Final Problem"", Sherlock Holmes finds ...",{'arthur conan doyle'},7,NaN,[],title_author,,Fiction,Fiction,"In ""The Final Problem"", Sherlock Holmes finds ..."
6,l4MZAAAAYAAJ,A House-boat on the Styx,NaN,John Kendrick Bangs,NaN,1898,The premise of the book is that everyone who h...,Fiction,238.0,en,...,The premise of the book is that everyone who h...,{'john kendrick bangs'},4,"The souls of the dead must cross the Styx, to ...","[""Fiction"", ""Houseboats"", ""Fiction, fantasy, h...",title_author,"Fiction, Houseboats, Fiction, fantasy, histori...","Fiction, Houseboats, fantasy, historical, coll...","Houseboats, fantasy, historical, collections &...",The premise of the book is that everyone who h...
7,Rf2dEAAAQBAJ,Virgin Soil,in large print,Ivan Sergeevich Turgenev,BoD – Books on Demand,2022-11-25,Reproduction of the original.,Fiction,429.0,en,...,Reproduction of the original.,{'ivan sergeevich turgenev'},7,NaN,[],title_author,,Fiction,Fiction,Reproduction of the original.
8,69JaAAAAMAAJ,Hemingway,A Life Without Consequences,James R. Mellow,NaN,1992,"Summary: A biography of the legendary writer, ...",Biography & Autobiography,742.0,en,...,"Summary: A biography of the legendary writer, ...",{'james r mellow'},4,"Summary: A biography of the legendary writer, ...","[""American Novelists"", ""Biography"", ""Hemingway...",title_author,"American Novelists, Biography, Hemingway, Erne...","Biography & Autobiography, American Novelists,...","Biography & Autobiography, American Novelist